In [ ]:
from dotenv import load_dotenv
load_dotenv('../../.env') ## load api key from .env file
import os,sys
sys.path.insert(0, '../../libs')
import pydantic
from pydantic import BaseModel, Field
from typing import List
from llm_factory_openai import LLMAgent


#### Basic api connection test 

In [3]:
llm_agent = LLMAgent(api_key=os.getenv('OPENAI_API_KEY'),
               model='gpt-5-nano',
               temperature=1) ## for gpt 5 model, temperature can only be 1

# test connection
print(llm_agent.test_connection())

# Example direct API call
print("\nExample direct API call:")
messages = [{"role": "user", "content": "What is 2+2?"}]
response = llm_agent._make_api_call(messages,reasoning_effort='low')
print(f"Response: {response.choices[0].message.content}")


Hi! Yes, I can respond. How can I help you today? I can answer questions, explain topics, help with writing or coding, do math, plan things, translate text, and more.

If you’re not sure where to start, here are a few examples:
- Explain how photosynthesis works in simple terms
- Draft a short email or message
- Write a Python function to sort a list
- Summarize a paragraph or article
- Plan a 3-day trip to a city
- Translate a sentence into another language

Tell me what you’d like to do.

Example direct API call:
Response: 4


In [4]:
paragraph = """The assessment of cyclical position and economic outlook involves evaluating the current and projected state of an economy over various time horizons. 
This includes analyzing near-term and medium-term growth prospects, 
understanding the business cycle phases (expansion and contraction), 
and identifying potential economic risks and uncertainties. 
Key indicators such as GDP growth, inflation, and the output gap are scrutinized to gauge macroeconomic stability. 
The evaluation also considers the impacts of fiscal and monetary policies on economic trends and forecasts potential scenarios, 
highlighting recession risks and opportunities for economic expansion."""
messages=[{"role": "system",
"content": '''You are an experience macroeconomist from IMF. Your job is to assign topic labels to a given paragraph from IMF document.
You are given a list of topics with their definition and key indicators as below:
----------------
----------------
1. **Economic Outlook**:
- **Definition**: The assessment of cyclical position and economic outlook involves evaluating the current and projected state of an economy over various time horizons. This includes analyzing near-term and medium-term growth prospects, understanding the business cycle phases (expansion and contraction), and identifying potential economic risks and uncertainties. Key indicators such as GDP growth, inflation, and the output gap are scrutinized to gauge macroeconomic stability. The evaluation also considers the impacts of fiscal and monetary policies on economic trends and forecasts potential scenarios, highlighting recession risks and opportunities for economic expansion.
- **Key Indicators**: economic outlook, near-term growth, medium-term growth, economic assessment, GDP growth, business cycle, economic forecast, projected growth, output gap, cyclical analysis, economic risks, economic indicators, macroeconomic stability, recession risk, expansion phase, contraction phase, economic trends

2. **Monetary Policy**:
- **Definition**: Monetary policy refers to the actions undertaken by a central bank, such as the Federal Reserve or the European Central Bank, to manage the economy by controlling the money supply, interest rates, and inflation. It aims to achieve price stability, full employment, and economic growth. Key aspects include setting the policy rate, managing inflation expectations and targets, addressing inflationary pressures, and ensuring financial stability. Monetary policy can involve conventional measures, such as adjusting interest rates, and unconventional tools, like quantitative easing and monetary tightening. It also encompasses the monetary transmission mechanism, which describes how policy actions affect the economy, and the interaction with fiscal policy.
- **Key Indicators**: inflation expectations, inflation target, inflationary pressures, monetary policy stance, policy rate, price stability, interest rates, central bank, quantitative easing, monetary tightening, unconventional monetary policy, monetary transmission mechanism, currency exchange rates, liquidity management, money supply, aggregate demand

3. **Fiscal Stance**:
- **Definition**: The fiscal stance and debt topic encompasses the analysis and evaluation of a government's fiscal policies and their impact on economic sustainability. This includes assessing fiscal sustainability, consolidation efforts, and the overarching fiscal framework that guides policy decisions. Key considerations involve the management of fiscal space, budget allocations, and the balance between fiscal deficits and surpluses. The topic also examines the influence of oil and non-oil revenues on fiscal health, the intricacies of managing government debt, and strategies for ensuring debt sustainability. Understanding the relationship between fiscal policy, expenditure, GDP, and various forms of debt (public, external, and domestic) is crucial for formulating effective economic strategies and maintaining financial stability.
- **Key Indicators**: fiscal sustainability, fiscal consolidation, fiscal framework, fiscal policy, fiscal space, budget, fiscal deficit, primary deficit, balanced budget, fiscal stance, oil revenue, non-oil revenue, government debt, expenditure, debt sustainability, debt management, external debt, public debt, domestic debt

4. **Financial Stability**:
- **Definition**: Financial stability refers to the resilience of the financial system, including banks, financial markets, and other financial institutions, in withstanding economic shocks and maintaining efficient functioning. It encompasses various aspects such as financial inclusion, risk management, credit growth, and the health of the banking sector. Key elements include the implementation of macroprudential policies, management of non-performing loans (NPLs), maintenance of adequate capital and liquidity levels, and robust supervision and stress testing of financial institutions. Effective governance, rigorous internal and external audits, and adherence to reporting standards and safeguards assessments are essential to ensure financial stability. Regulatory measures and assessment recommendations play a crucial role in sustaining the overall health of the financial system.
- **Key Indicators**: financial inclusion, financial stability, risk, banking sector, credit growth, financial institutions, macroprudential, non-performing loans (NPLs), capital, credit risk, liquidity, supervision, stress tests, bank governance, internal audit, reporting standards, safeguards assessment, external audit, assessment recommendations, regulatory measures

5. **External Stance**:
- **Definition**: The topic covers the macroeconomic analysis of a country's external economic health and the dynamics of its currency's exchange rate. Key elements include international reserves, current account deficits and surpluses, and the effective exchange rate, which are essential indicators of external balance. The topic also encompasses the analysis of implied fundamentals through models like the EBA-lite, and the impact of capital grants, trade policies, and trade elasticities on the external position. Understanding trade liberalization, integration, and barriers further helps in evaluating the broader context of global trade influences on exchange rates.
- **Key Indicators**: international reserves, current account deficit, implied fundamentals, external balance, effective exchange rate, current account surplus, EBA-lite model, external deficit, capital grants, exchange rate, import elasticity, export elasticity, trade liberalization, trade policy, trade integration, bilateral trade, global trade, tariff barriers, foreign direct investment (FDI)
----------------
----------------

Given a paragraph from a report published by the International Monetary Fund, please carefully analyze the paragraph and classify the provided paragraph using ONLY the provided topics. If the paragraph does not fit into any of the provided topics, assign "Other".
Try your best to assign only one topic to the paragraph. You can use multiple categories only if you are very confident that multiple topics are all extensively discussed by the majority of the paragraph.
Please provide your reasoning for your classification first, and then provide the topic label and a confidence score from 0-100.

Please respond in clean json format as follow:
reasoning: <explanation for topic label >,
topic_labels: [<topic label: confidence score>,...]''',},
{   "role": "user",
    "content": paragraph}
            ]

### define response model and get structured output

In [8]:
class TopicLabel(BaseModel):
    topic: str = Field(..., description="Topic label from the predefined list")
    confidence: int = Field(..., description="Confidence score from 0-100", ge=0, le=100)

class TopicResponse(BaseModel):
    reasoning: str = Field(..., description="Explanation for the topic classification")
    topic_labels: List[TopicLabel] = Field(..., description="List of topic labels with confidence scores")


In [17]:
response=llm_agent.get_response_content(messages,reasoning_effort='low',response_format=TopicResponse)
print('reasoning:',response.reasoning)
for topic in response.topic_labels:
    print(f"{topic.topic}: {topic.confidence}")

reasoning: The paragraph discusses assessing cyclical position and the economic outlook, including near-term and medium-term growth, business cycle phases, risks, and key indicators like GDP growth, inflation, and the output gap. It also notes the impact of fiscal and monetary policy on trends and forecasts, which aligns with the Economic Outlook topic rather than separate Monetary Policy or Fiscal Stance specifics.
Economic Outlook: 90
